In [2]:
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer
import os
import time
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.pre_tokenizers import ByteLevel
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

MODEL_NAME = "openai-community/gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, device_map="auto")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
import inspect
inspect.getsourcefile(type(tokenizer))

'/home/newro/projects/ohmyllm/.venv/lib/python3.13/site-packages/transformers/models/gpt2/tokenization_gpt2.py'

In [14]:
tokenizer.unk_token, tokenizer.eos_token, tokenizer.pad_token

('<|endoftext|>', '<|endoftext|>', None)

In [ ]:
def compare_token_efficiency(tokenizer: AutoTokenizer, text_en: str, text_ko: str):
    en_ids = tokenizer.encode(text_en, add_special_tokens=False)
    ko_ids = tokenizer.encode(text_ko, add_special_tokens=False)
    result = {
        "english_text": text_en,
        "korean_text": text_ko,
        "english_tokens": len(en_ids),
        "korean_tokens": len(ko_ids),
        "tokens_per_char_en": len(en_ids) / max(len(text_en), 1),
        "tokens_per_char_ko": len(ko_ids) / max(len(text_ko), 1),
        "token_ratio_ko_to_en": len(ko_ids) / max(len(en_ids), 1),
    }
    return result


sample_en = "Hello, how are you today? I hope you're doing well."
sample_ko = "안녕하세요, 오늘 기분이 어때요? 잘 지내고 있길 바라요."


result = compare_token_efficiency(tokenizer, sample_en, sample_ko)

print("Token Efficiency Comparison")
print("-" * 32)
print(f"English text: {result['english_text']}")
print(f"Korean text : {result['korean_text']}")
print()
print(f"English tokens      : {result['english_tokens']}")
print(f"Korean tokens       : {result['korean_tokens']}")
print(
    f"Tokens/char (EN)    : {result['tokens_per_char_en'] * 100:.1f}% (영문 문자당 토큰 비율)"
)
print(
    f"Tokens/char (KO)    : {result['tokens_per_char_ko'] * 100:.1f}% (한글 문자당 토큰 비율)"
)
print(
    f"KO/EN token ratio   : {result['token_ratio_ko_to_en'] * 100:.1f}% (한국어 토큰 수/영어 토큰 수 비율)"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Token Efficiency Comparison
--------------------------------
English text: Hello, how are you today? I hope you're doing well.
Korean text : 안녕하세요, 오늘 기분이 어때요? 잘 지내고 있길 바라요.

English tokens      : 14
Korean tokens       : 68
Tokens/char (EN)    : 27.5% (영문 문자당 토큰 비율)
Tokens/char (KO)    : 212.5% (한글 문자당 토큰 비율)
KO/EN token ratio   : 485.7% (한국어 토큰 수/영어 토큰 수 비율)


In [3]:
# 1) 말뭉치 로드 및 읽기 시간 측정
t0 = time.perf_counter()
with open("../data/corpus.txt", "r", encoding="utf-8") as f:
    corpus_text = f.read()
t1 = time.perf_counter()
print(f"Read corpus: {t1 - t0:.2f}s")

# 2) 스레드 개수/청크 크기 계산 후 텍스트를 분할
num_workers = max(1, (os.cpu_count() or 1) - 1)
chunk_size = max(1, len(corpus_text) // (num_workers * 4))
chunks = [
    corpus_text[i : i + chunk_size] for i in range(0, len(corpus_text), chunk_size)
]
print(f"Encoding with {num_workers} workers, {len(chunks)} chunks")


def encode_chunk(text_chunk: str):
    # 청크 단위로 토큰화하여 병렬 처리에 사용
    return tokenizer.encode(text_chunk, add_special_tokens=False)


# 3) 스레드풀로 청크 병렬 토큰화
chunk_ids = []
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(encode_chunk, chunk) for chunk in chunks]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Encoding"):
        chunk_ids.append(future.result())
token_ids = [token_id for chunk in chunk_ids for token_id in chunk]
t2 = time.perf_counter()
print(f"Tokenize (threads): {t2 - t1:.2f}s")

# 4) 토큰 빈도 집계
counts = Counter(token_ids)
t3 = time.perf_counter()
print(f"Count tokens: {t3 - t2:.2f}s")

# 5) 상위 토큰 출력
top_n = 50
top_tokens = counts.most_common(top_n)

print(f"Top {top_n} tokens")
print("-" * 40)
for token_id, freq in top_tokens:
    token_str = tokenizer.decode([token_id])
    print(f"{token_str}\t(id={token_id})\t{freq}")

Read corpus: 1.49s
Encoding with 13 workers, 53 chunks


Encoding:   0%|          | 0/53 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (633685 > 1024). Running this sequence through the model will result in indexing errors


Tokenize (threads): 30.26s
Count tokens: 5.35s
Top 50 tokens
----------------------------------------
�	(id=168)	19106192
�	(id=167)	15181069
 �	(id=23821)	11700664
�	(id=166)	11114573
�	(id=226)	8257099
�	(id=250)	7820960
�	(id=108)	7441932
�	(id=246)	7381995
 	(id=220)	7328446
�	(id=112)	6821467
�	(id=35975)	6643636
�	(id=254)	6539761
 �	(id=31619)	6387884
�	(id=222)	6290978
�	(id=169)	5058832
�	(id=230)	5042961
�	(id=116)	4772056
�	(id=97)	4662656
�	(id=47991)	4369856
�	(id=238)	4361356
�	(id=120)	4254655
�	(id=242)	4018096
�	(id=46695)	3572108
�	(id=111)	3478723
�	(id=234)	3475257
�	(id=232)	3336396
�	(id=245)	3218643
�	(id=252)	3215119
�	(id=251)	3071037
�	(id=105)	2993999
�	(id=243)	2975204
�	(id=100)	2857755
�	(id=110)	2495929
�	(id=113)	2364150
�	(id=223)	2068782
�	(id=94)	2038863
�	(id=98)	1965171
�	(id=233)	1909828
�	(id=247)	1892164
.	(id=13)	1848645
�	(id=102)	1772439
�	(id=244)	1699911
�	(id=248)	1698905
�	(id=225)	1645392
�	(id=227)	1546801
�	(id=117)	1471211
�	(id=237)	1

In [4]:
# 저빈도 토큰 찾기 (예: 빈도 <= 1)
low_freq_threshold = 1
max_show = 50

low_freq_tokens = [
    (token_id, freq) for token_id, freq in counts.items() if freq <= low_freq_threshold
]

low_freq_tokens = sorted(low_freq_tokens, key=lambda x: (x[1], x[0]))

print(f"Low-frequency tokens (freq <= {low_freq_threshold})")
print("-" * 40)
for token_id, freq in low_freq_tokens[:max_show]:
    token_str = tokenizer.decode([token_id])
    print(f"{token_str}\t(id={token_id})\t{freq}")

print(f"\nTotal low-frequency tokens: {len(low_freq_tokens)}")

Low-frequency tokens (freq <= 1)
----------------------------------------
�	(id=131)	1
	(id=208)	1
	(id=219)	1
clud	(id=758)	1
iew	(id=769)	1
meric	(id=946)	1
ween	(id=975)	1
 Americ	(id=1019)	1
 mem	(id=1066)	1
 contro	(id=1246)	1
 contin	(id=1261)	1
riend	(id=1289)	1
 adv	(id=1354)	1
 beh	(id=1372)	1
ocr	(id=1696)	1
 econom	(id=1707)	1
aign	(id=1784)	1
 polic	(id=1825)	1
 hig	(id=1880)	1
 ey	(id=1926)	1
 sugg	(id=1947)	1
 vot	(id=1993)	1
 trad	(id=2083)	1
ederal	(id=2110)	1
 benef	(id=2204)	1
 indust	(id=2226)	1
 coun	(id=2289)	1
 Canad	(id=2294)	1
 sus	(id=2341)	1
ilities	(id=2410)	1
 compan	(id=2463)	1
ivil	(id=2464)	1
itle	(id=2578)	1
apan	(id=2674)	1
 indic	(id=2699)	1
ctor	(id=2715)	1
 defin	(id=2730)	1
 tou	(id=2819)	1
 stru	(id=2874)	1
hern	(id=2881)	1
 arri	(id=2914)	1
ription	(id=2918)	1
sych	(id=2924)	1
 behav	(id=2955)	1
 recomm	(id=3045)	1
ysis	(id=3097)	1
 ens	(id=3140)	1
ornia	(id=3317)	1
vertis	(id=3346)	1
nown	(id=3408)	1

Total low-frequency tokens: 4217


In [5]:
# 저빈도 토큰 수 만큼의 보캡(자주 등장하는 토큰) 추출 - tokenizers 기반 (Unigram)

# 1) 이전 모델의 UNK 토큰만 가져와 새 토크나이저에 포함
unk_token: str = tokenizer.unk_token
print(f"UNK token: {unk_token}")

# 2) Unigram 토크나이저를 코퍼스로 학습 (고빈도 토큰을 얻기 위한 용도)
replace_vocab_size = len(low_freq_tokens)
print(f"Target vocab size for replacement: {replace_vocab_size}")
unigram_tokenizer = Tokenizer(Unigram())
unigram_tokenizer.pre_tokenizer = ByteLevel()
trainer = UnigramTrainer(
    vocab_size=replace_vocab_size,
    show_progress=True,
    unk_token=unk_token,
    shrinking_factor=0.75,
    max_piece_length=16,
    n_sub_iterations=2,
)

# 진행바가 0/1로 고정되는 문제를 피하려고 작은 서브청크로 나눠서 학습
subchunk_size = 2000
print(f"Training Unigram tokenizer with subchunk_size={subchunk_size}...")


def iter_subchunks(text_chunks, size):
    # 긴 청크를 더 작은 조각으로 나눠 진행률을 촘촘히 표시
    for chunk in text_chunks:
        for i in range(0, len(chunk), size):
            yield chunk[i : i + size]


total_steps = sum((len(chunk) + subchunk_size - 1) // subchunk_size for chunk in chunks)
train_iter = tqdm(
    iter_subchunks(chunks, subchunk_size), total=total_steps, desc="Training (Unigram)"
)
unigram_tokenizer.train_from_iterator(train_iter, trainer=trainer)
print("Training done.")

UNK token: <|endoftext|>
Target vocab size for replacement: 4217
Training Unigram tokenizer with subchunk_size=2000...


Training (Unigram):   0%|          | 0/69525 [00:00<?, ?it/s]



Training done.


In [ ]:
# 3) 코퍼스를 새 토크나이저로 인코딩해서 실제 빈도 집계 (청크/배치 병렬)
batch_size = 32
batches = [chunks[i : i + batch_size] for i in range(0, len(chunks), batch_size)]
print(f"Encoding {len(batches)} batches with batch_size={batch_size}")

# 진행률을 더 촘촘히 표시하기 위해 배치를 더 작은 서브배치로 분할
sub_batch_size = 4
sub_batches = []
for batch in batches:
    sub_batches.extend([batch[i : i + sub_batch_size] for i in range(0, len(batch), sub_batch_size)])
print(f"Encoding {len(sub_batches)} sub-batches with sub_batch_size={sub_batch_size}")

def encode_batch(batch_texts):
    # 배치 단위로 Unigram 토크나이저로 인코딩
    encodings = unigram_tokenizer.encode_batch(batch_texts)
    batch_ids = []
    for enc in encodings:
        batch_ids.extend(enc.ids)
    return batch_ids

unigram_token_ids = []
print(f"Parallel encoding with {num_workers} workers")
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [executor.submit(encode_batch, batch) for batch in sub_batches]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Encoding (Unigram)"):
        unigram_token_ids.extend(future.result())
print(f"Total encoded token IDs: {len(unigram_token_ids)}")
unigram_counts = Counter(unigram_token_ids)
print(f"Unique Unigram tokens: {len(unigram_counts)}")

Encoding 2 batches with batch_size=32
Encoding 14 sub-batches with sub_batch_size=4
Parallel encoding with 13 workers


Encoding (Unigram):   0%|          | 0/14 [00:00<?, ?it/s]

Total encoded token IDs: 78963419
Unique Unigram tokens: 4214


In [8]:
# 4) 저빈도 토큰 수 만큼 고빈도 토큰 추출
replacement_tokens = unigram_counts.most_common(replace_vocab_size)
replacement_vocab = [
    {
        "token_id": token_id,
        "token_str": unigram_tokenizer.id_to_token(token_id),
        "freq": freq,
    }
    for token_id, freq in replacement_tokens
]

print(f"Replacement vocabulary (top {len(replacement_vocab)} tokens)")
print("-" * 40)
for item in replacement_vocab:
    print(f"{item['token_str']}\t(id={item['token_id']})\t{item['freq']}")
print()

Replacement vocabulary (top 4214 tokens)
----------------------------------------
Ġ	(id=1)	6159834
.	(id=2)	1886224
,	(id=3)	1425631
Ċ	(id=4)	1231535
ìĿĺ	(id=5)	1149311
ìĿĦ	(id=6)	1030130
ìĿ´	(id=7)	955617
ìĹĲ	(id=8)	783077
ëĬĶ	(id=9)	701109
ê°Ģ	(id=10)	697206
ìĿĢ	(id=11)	606624
ë¥¼	(id=12)	536766
ê³ł	(id=13)	436958
ëıĦ	(id=14)	426029
s	(id=16)	409495
ê³¼	(id=17)	390883
ĠìĪĺ	(id=18)	370954
íķľ	(id=15)	369699
ì§Ģ	(id=19)	352148
ë¡ľ	(id=20)	346044
ê¸°	(id=21)	321745
ìĿ¸	(id=22)	306870
(	(id=23)	294532
ìĹĲìĦľ	(id=24)	283897
ĠìĿ´	(id=36)	270992
íķĺê³ł	(id=30)	270909
)	(id=25)	266546
?	(id=26)	264556
ìľ¼ë¡ľ	(id=27)	255795
"	(id=28)	249499
ìĻĢ	(id=31)	237873
íķĺëĬĶ	(id=29)	229642
ìĿ¼	(id=32)	228641
ë¦¬	(id=34)	223277
ëĭ¤	(id=33)	196519
Ġthe	(id=38)	195483
íķł	(id=35)	190677
Ġê·¸	(id=40)	186594
ìĦľ	(id=42)	183100
íķ´	(id=37)	179669
ìĤ¬	(id=41)	170351
ìŀĲ	(id=39)	160407
ìĸ´	(id=44)	154996
ĠìŀĪìĬµëĭĪëĭ¤	(id=47)	154917
ëĮĢ	(id=46)	154019
ìĭľ	(id=43)	153681
ë§Į	(id=48)	153152
ìļĶ	(id=52)	150285
ì

In [11]:
# 5) 저빈도 토큰을 고빈도 토큰으로 치환한 새 Unigram 토크나이저 구성
import math

# 기존 보캡 정렬 (id 순서 유지)
vocab_items = sorted(tokenizer.get_vocab().items(), key=lambda x: x[1])
vocab_size = len(vocab_items)
id_to_token = [token for token, _ in vocab_items]
token_to_id = {token: token_id for token, token_id in vocab_items}

# 치환 대상 id 목록 (특수 토큰은 보호)
protected_ids = set(getattr(tokenizer, "all_special_ids", []))
if tokenizer.unk_token_id is not None:
    protected_ids.add(tokenizer.unk_token_id)
low_freq_ids = [
    token_id for token_id, _ in low_freq_tokens if token_id not in protected_ids
 ]
print(f"Low-frequency ids (replaceable): {len(low_freq_ids)}")

# 1) 필요한 개수만큼 고빈도 토큰을 먼저 선택 (중복/기존 보캡 제외)
replacement_needed = len(low_freq_ids)
replacement_tokens_filtered = []
replacement_seen = set()
existing_token_set = set(id_to_token)
for token_id, _ in unigram_counts.most_common():
    token_str = unigram_tokenizer.id_to_token(token_id)
    if token_str in existing_token_set or token_str in replacement_seen:
        continue
    replacement_tokens_filtered.append(token_str)
    replacement_seen.add(token_str)
    if len(replacement_tokens_filtered) >= replacement_needed:
        break
print(f"Replacement tokens selected: {len(replacement_tokens_filtered)}")
if len(replacement_tokens_filtered) < replacement_needed:
    print("Warning: not enough unique replacement tokens; some low-frequency tokens will remain.")

# 2) 선택한 만큼만 치환, 나머지는 원래 토큰 유지 (보캡 크기/ID 유지)
new_id_to_token = list(id_to_token)
for idx, token_id in enumerate(low_freq_ids):
    if idx < len(replacement_tokens_filtered):
        new_id_to_token[token_id] = replacement_tokens_filtered[idx]
    else:
        new_id_to_token[token_id] = id_to_token[token_id]
print(f"Original vocab size: {vocab_size}")
print(f"Final vocab size: {len(new_id_to_token)}")

# Unigram 점수 계산 (log-prob)
token_freq_pairs = []
for token_id, token_str in enumerate(new_id_to_token):
    if token_str in token_to_id:
        # 기존 보캡 토큰은 원래 id의 빈도를 사용
        base_id = token_to_id[token_str]
        freq = counts.get(base_id, 0)
    else:
        # 신규 치환 토큰은 Unigram 학습 빈도를 사용
        rep_id = unigram_tokenizer.token_to_id(token_str)
        freq = unigram_counts.get(rep_id, 0) if rep_id is not None else 0
    token_freq_pairs.append((token_str, freq))
total_freq = sum(freq for _, freq in token_freq_pairs)
total_freq = total_freq if total_freq > 0 else 1
new_vocab = [
    (token, math.log((freq + 1) / total_freq)) for token, freq in token_freq_pairs
 ]

# UNK 토큰 보장 및 위치 지정 (기존 id 유지 우선)
unk_token = tokenizer.unk_token or "<unk>"
if tokenizer.unk_token_id is not None:
    unk_id = tokenizer.unk_token_id
    if new_vocab[unk_id][0] != unk_token:
        new_vocab[unk_id] = (unk_token, new_vocab[unk_id][1])
else:
    if unk_token not in {token for token, _ in new_vocab}:
        new_vocab.insert(0, (unk_token, math.log(1 / total_freq)))
    unk_id = next(i for i, (tok, _) in enumerate(new_vocab) if tok == unk_token)

new_unigram_tokenizer = Tokenizer(Unigram(new_vocab, unk_id=unk_id))
new_unigram_tokenizer.pre_tokenizer = ByteLevel()
print("New Unigram tokenizer created.")

output_dir = "./data/adapted_tokenizer"
os.makedirs(output_dir, exist_ok=True)
tokenizer_path = os.path.join(output_dir, "unigram_tokenizer.json")
new_unigram_tokenizer.save(tokenizer_path)
print(f"Adapted tokenizer saved to: {tokenizer_path}")

Low-frequency ids (replaceable): 4217
Replacement tokens selected: 3407
Original vocab size: 50257
Final vocab size: 50257
New Unigram tokenizer created.
Adapted tokenizer saved to: ./data/adapted_tokenizer/unigram_tokenizer.json


In [43]:
enc

[Encoding(num_tokens=14, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=11, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=10, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [53]:
# 6) 새 토크나이저 vs 이전 토크나이저: 한국어/영어 배치 비교
from typing import List, Dict

from tokenizers import Encoding

en_texts = [
    "Hello, how are you today? I hope you're doing well.",
    "This is a short English sentence for tokenizer comparison.",
    "We are testing batch tokenization with multiple samples.",
    "Tokenization efficiency matters for longer inputs.",
]
ko_texts = [
    "안녕하세요, 오늘 기분이 어때요? 잘 지내고 있길 바라요.",
    "토크나이저 비교를 위한 짧은 한국어 문장입니다.",
    "여러 샘플을 배치로 토큰화하는 테스트를 진행합니다.",
    "긴 입력에서 토큰화 효율이 중요합니다.",
]


def stats_gpt2(texts: List[str]) -> Dict[str, float]:
    backend_tokenizer: Tokenizer = tokenizer.backend_tokenizer
    encodings: list[Encoding] = backend_tokenizer.encode_batch(texts)
    token_counts = [len(encoding.ids) for encoding in encodings]
    char_counts = [len(t) for t in texts]
    total_tokens = sum(token_counts)
    total_chars = sum(char_counts)
    return {
        "total_tokens": total_tokens,
        "total_chars": total_chars,
        "tokens_per_char": total_tokens / max(total_chars, 1),
        "avg_tokens_per_text": total_tokens / max(len(texts), 1),
    }


def stats_unigram(texts: List[str]) -> Dict[str, float]:
    encodings: list[Encoding] = new_unigram_tokenizer.encode_batch(texts)
    token_counts = [len(e.ids) for e in encodings]
    char_counts = [len(t) for t in texts]
    total_tokens = sum(token_counts)
    total_chars = sum(char_counts)
    return {
        "total_tokens": total_tokens,
        "total_chars": total_chars,
        "tokens_per_char": total_tokens / max(total_chars, 1),
        "avg_tokens_per_text": total_tokens / max(len(texts), 1),
    }


def make_row(
    label: str, old_stats: Dict[str, float], new_stats: Dict[str, float]
) -> str:
    ratio = new_stats["total_tokens"] / max(old_stats["total_tokens"], 1)
    return (
        f"| {label} | {old_stats['total_tokens']} | {old_stats['tokens_per_char']:.4f} | {old_stats['avg_tokens_per_text']:.2f} | "
        f"{new_stats['total_tokens']} | {new_stats['tokens_per_char']:.4f} | {new_stats['avg_tokens_per_text']:.2f} | {ratio:.4f} |"
    )


def print_table(en_stats_old, en_stats_new, ko_stats_old, ko_stats_new):
    print(
        "| Language | Old tokens | Old tok/char | Old avg/text | New tokens | New tok/char | New avg/text | New/Old |"
    )
    print("|---|---:|---:|---:|---:|---:|---:|---:|")
    print(make_row("English", en_stats_old, en_stats_new))
    print(make_row("Korean", ko_stats_old, ko_stats_new))


en_old = stats_gpt2(en_texts)
en_new = stats_unigram(en_texts)
ko_old = stats_gpt2(ko_texts)
ko_new = stats_unigram(ko_texts)
print_table(en_old, en_new, ko_old, ko_new)


# 7) 모델 실제 출력 (기존 GPT-2 토크나이저 기준, 배치 처리)
def generate_batch(texts: List[str], max_new_tokens: int = 30):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
    )
    return tokenizer.batch_decode(output_ids, skip_special_tokens=True)


gen_en = generate_batch(en_texts)
gen_ko = generate_batch(ko_texts)

print("\n[Model outputs - English]")
for i, text in enumerate(gen_en, 1):
    print(f"{i}. {text}")

print("\n[Model outputs - Korean]")
for i, text in enumerate(gen_ko, 1):
    print(f"{i}. {text}")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


| Language | Old tokens | Old tok/char | Old avg/text | New tokens | New tok/char | New avg/text | New/Old |
|---|---:|---:|---:|---:|---:|---:|---:|
| English | 43 | 0.2000 | 10.75 | 43 | 0.2000 | 10.75 | 1.0000 |
| Korean | 235 | 2.1963 | 58.75 | 78 | 0.7290 | 19.50 | 0.3319 |


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.



[Model outputs - English]
1. Hello, how are you today? I hope you're doing well. I'll give you a call tomorrow, and you'll tell me how much you're enjoying this. And then you can tell me how much you're
2. This is a short English sentence for tokenizer comparison.CatchTokenizer() is called when a tokenizer is used.

Returns the amount of tokenized tokens that can be added to the system
3. We are testing batch tokenization with multiple samples.If you are not familiar with the basic idea of tokens, it is a general rule to use a number of different tokens (in this case, 3
4. Tokenization efficiency matters for longer inputs.

Here's a sample of how this might work. If we're using a client that accepts an API key, we'll create an instance of

[Model outputs - Korean]
1. 안녕하세요, 오늘 기분이 어때요? 잘 지내고 있길 바라요. 두지정에에에 배고 스타
2. 토크나이저 비교를 위한 짧은 한국어 문장입니다.

로 장마는텔로 스텔로 
3. 여러 샘플을 배치로 토큰화하는 테스트를 진행합니다.

바리 다이요인 바리 시들
4. 긴 입력에서 토큰화 효율이 중요합니다.

하서는도 가장 세요은 �


In [ ]:
# 7) 코퍼스 + 새 토크나이저로 임베딩만 미세 조정: 토크나이저/모델 준비
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast

new_fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=tokenizer_path,
    unk_token=unk_token,
    eos_token=tokenizer.eos_token,
 )
if new_fast_tokenizer.pad_token is None:
    new_fast_tokenizer.pad_token = new_fast_tokenizer.eos_token

model.resize_token_embeddings(len(new_fast_tokenizer))
for param in model.parameters():
    param.requires_grad = False
model.transformer.wte.weight.requires_grad = True

In [ ]:
# 데이터셋 구성
block_size = 128
max_tokens = 50000
all_ids = new_fast_tokenizer.encode(corpus_text)[:max_tokens]
blocks = [
    all_ids[i : i + block_size]
    for i in range(0, len(all_ids) - block_size, block_size)
 ]
print(f"Blocks for training: {len(blocks)}")

class BlockDataset(Dataset):
    def __init__(self, blocks_list):
        self.blocks_list = blocks_list
    def __len__(self):
        return len(self.blocks_list)
    def __getitem__(self, idx):
        return torch.tensor(self.blocks_list[idx], dtype=torch.long)

train_loader = DataLoader(BlockDataset(blocks), batch_size=8, shuffle=True)
optimizer = torch.optim.AdamW([model.transformer.wte.weight], lr=5e-4)

In [ ]:
# 학습 루프
model.train()
max_steps = 200
step = 0
for batch in train_loader:
    batch = batch.to(model.device)
    outputs = model(input_ids=batch, labels=batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    step += 1
    if step % 20 == 0:
        print(f"Step {step}/{max_steps} - loss: {loss.item():.4f}")
    if step >= max_steps:
        break

model.eval()

In [ ]:
# 새 토크나이저로 생성 테스트
def generate_with_new_tokenizer(texts, max_new_tokens=30):
    inputs = new_fast_tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
    )
    return new_fast_tokenizer.batch_decode(output_ids, skip_special_tokens=True)

gen_en_new = generate_with_new_tokenizer(en_texts[:2])
gen_ko_new = generate_with_new_tokenizer(ko_texts[:2])

print("\n[Adapted tokenizer outputs - English]")
for i, text in enumerate(gen_en_new, 1):
    print(f"{i}. {text}")

print("\n[Adapted tokenizer outputs - Korean]")
for i, text in enumerate(gen_ko_new, 1):
    print(f"{i}. {text}")